In [1]:
#!/usr/bin/env python3
"""
Генератор датасета цифр для CNN. 50k изображений 224x224.
Многопоточный (threading), с логированием.
"""

import os, io, random, math, time
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# ─── Конфигурация ─────────────────────────────────────────────────────────────
IMG_SIZE         = 224
IMAGES_PER_CLASS = 5000
OUTPUT_DIR       = "dataset"
NUM_WORKERS      = 10

# ─── Логгер ───────────────────────────────────────────────────────────────────

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

# ─── Шрифты ───────────────────────────────────────────────────────────────────

def _find_fonts():
    base = r"C:\Windows\Fonts"
    safe = [
        "arial.ttf", "arialbd.ttf", "ariali.ttf", "ariblk.ttf",
        "calibri.ttf", "calibrib.ttf", "calibrii.ttf",
        "verdana.ttf", "verdanab.ttf", "verdanai.ttf",
        "tahoma.ttf", "tahomabd.ttf",
        "segoeui.ttf", "segoeuib.ttf", "segoeuil.ttf",
        "trebuc.ttf", "trebucbd.ttf",
        "corbel.ttf", "corbelb.ttf",
        "Candara.ttf", "Candarab.ttf",
        "bahnschrift.ttf", "impact.ttf",
        "AGENCYB.TTF", "AGENCYR.TTF",
        "GOTHIC.TTF", "GOTHICB.TTF",
        "MAIAN.TTF", "DUBAI-BOLD.TTF", "DUBAI-REGULAR.TTF",
        "times.ttf", "timesbd.ttf", "timesi.ttf",
        "georgia.ttf", "georgiab.ttf",
        "pala.ttf", "palab.ttf",
        "BOOKOS.TTF", "BOOKOSB.TTF",
        "CENTURY.TTF", "BASKVILL.TTF",
        "GARA.TTF", "GARABD.TTF",
        "BELL.TTF", "BELLB.TTF",
        "BOD_R.TTF", "BOD_B.TTF",
        "ROCK.TTF", "ROCKB.TTF",
        "CENTAUR.TTF", "ELEPHNT.TTF",
        "cour.ttf", "courbd.ttf",
        "consola.ttf", "consolab.ttf",
        "CascadiaCode.ttf", "CascadiaMono.ttf",
        "lucon.ttf", "OCRAEXT.TTF",
        "comic.ttf", "comicbd.ttf",
        "BAUHS93.TTF", "BROADW.TTF", "BRITANIC.TTF",
        "STENCIL.TTF", "COOPBL.TTF",
        "ERASBD.TTF", "ERASMD.TTF",
        "SHOWG.TTF", "RAVIE.TTF", "PLAYBILL.TTF",
        "FORTE.TTF", "MOD20.TTF", "COLONNA.TTF", "ONYX.TTF",
        "Inkfree.ttf", "MISTRAL.TTF", "BRUSHSCI.TTF", "RAGE.TTF",
    ]
    valid = [os.path.join(base, f) for f in safe if os.path.exists(os.path.join(base, f))]
    if not valid:
        valid = [None]
    return valid

# ─── Глобальный список шрифтов ────────────────────────────────────────────────
FONTS = []

# ─── Утилиты ──────────────────────────────────────────────────────────────────

def rand_color():
    return (random.randint(0,255), random.randint(0,255), random.randint(0,255))

def contrasting_color(bg, min_delta=100):
    br = 0.299*bg[0] + 0.587*bg[1] + 0.114*bg[2]
    base = (random.randint(0, max(0, int(br)-min_delta)) if br > 128
            else random.randint(min(255, int(br)+min_delta), 255))
    return tuple(max(0,min(255, base+random.randint(-40,40))) for _ in range(3))

def make_background(size):
    style = random.choices(
        ["solid","grad_lin","grad_rad","noise","grid","stripes","dots"],
        weights=[15,20,10,25,10,10,10])[0]
    arr = np.zeros((size,size,3), dtype=np.uint8)

    if style == "solid":
        arr[:] = rand_color()
    elif style == "grad_lin":
        c1,c2 = np.array(rand_color(),dtype=np.float32), np.array(rand_color(),dtype=np.float32)
        angle = random.uniform(0, math.pi)
        xx,yy = np.meshgrid(np.linspace(0,1,size), np.linspace(0,1,size))
        t = xx*math.cos(angle)+yy*math.sin(angle)
        t = (t-t.min())/(t.max()-t.min()+1e-8)
        for ch in range(3): arr[:,:,ch]=(c1[ch]*(1-t)+c2[ch]*t).astype(np.uint8)
    elif style == "grad_rad":
        c1,c2 = np.array(rand_color(),dtype=np.float32), np.array(rand_color(),dtype=np.float32)
        cx,cy = random.uniform(0.2,0.8), random.uniform(0.2,0.8)
        xx,yy = np.meshgrid(np.linspace(0,1,size)-cx, np.linspace(0,1,size)-cy)
        t = np.clip(np.sqrt(xx**2+yy**2), 0, 1)
        t = t/t.max()
        for ch in range(3): arr[:,:,ch]=(c1[ch]*(1-t)+c2[ch]*t).astype(np.uint8)
    elif style == "noise":
        if random.random()<0.5:
            arr = np.random.randint(0,256,(size,size,3),dtype=np.uint8)
        else:
            bc = rand_color(); s = random.randint(20,80)
            for ch in range(3):
                arr[:,:,ch]=np.clip(np.random.normal(bc[ch],s,(size,size)),0,255).astype(np.uint8)
    elif style == "grid":
        arr[:]=rand_color(); lc=np.array(rand_color()); step=random.randint(12,40)
        for i in range(0,size,step): arr[i,:]=lc; arr[:,i]=lc
    elif style == "stripes":
        c1,c2=np.array(rand_color()),np.array(rand_color()); w=random.randint(8,40)
        vert=random.random()<0.5
        for i in range(size):
            col=c1 if (i//w)%2==0 else c2
            if vert: arr[:,i]=col
            else:    arr[i,:]=col
    elif style == "dots":
        bc,dc=rand_color(),rand_color(); arr[:]=bc
        img=Image.fromarray(arr,"RGB"); draw=ImageDraw.Draw(img)
        sp=random.randint(10,30); r=random.randint(1,sp//3)
        for y in range(0,size,sp):
            for x in range(0,size,sp):
                draw.ellipse([x-r,y-r,x+r,y+r],fill=dc)
        if random.random()<0.3:
            img=img.filter(ImageFilter.GaussianBlur(random.uniform(0.5,2)))
        return img

    img = Image.fromarray(arr,"RGB")
    if random.random()<0.3:
        img=img.filter(ImageFilter.GaussianBlur(random.uniform(0.5,2)))
    return img

def get_font(size):
    path = random.choice(FONTS)
    if path is None: return ImageFont.load_default()
    try:    return ImageFont.truetype(path, size)
    except: return ImageFont.load_default()

def _persp_coeffs(w, h, distort):
    d = distort*min(w,h)
    src=[(0,0),(w,0),(w,h),(0,h)]
    dst=[(random.uniform(-d,d),random.uniform(-d,d)),
         (w+random.uniform(-d,d),random.uniform(-d,d)),
         (w+random.uniform(-d,d),h+random.uniform(-d,d)),
         (random.uniform(-d,d),h+random.uniform(-d,d))]
    M=[]
    for (x,y),(X,Y) in zip(dst,src):
        M+=[[x,y,1,0,0,0,-X*x,-X*y],[0,0,0,x,y,1,-Y*x,-Y*y]]
    try:
        B=np.array([X for X,_ in src]+[Y for _,Y in src],dtype=np.float64)
        return np.linalg.solve(np.array(M,dtype=np.float64),B).tolist()
    except: return [1,0,0,0,1,0,0,0]

def render_digit(digit_str, sz=IMG_SIZE):
    bg = make_background(sz)
    patch = np.array(bg)[sz//4:3*sz//4, sz//4:3*sz//4]
    avg_bg = tuple(patch.mean(axis=(0,1)).astype(int))

    fnt = get_font(random.randint(80,180))
    dc  = contrasting_color(avg_bg, random.randint(70,160))

    pad   = sz
    total = sz+2*pad
    layer = Image.new("RGBA",(total,total),(0,0,0,0))
    draw  = ImageDraw.Draw(layer)

    try:    bb=fnt.getbbox(digit_str); tw,th=bb[2]-bb[0],bb[3]-bb[1]
    except: tw,th=fnt.getsize(digit_str)

    cx=total//2-tw//2+random.randint(-sz//8,sz//8)
    cy=total//2-th//2+random.randint(-sz//8,sz//8)

    sm=random.choices(["none","shadow","outline"],weights=[50,30,20])[0]
    if sm=="shadow":
        sc=contrasting_color(dc,60); off=random.randint(2,8)
        draw.text((cx+off,cy+off),digit_str,font=fnt,fill=(*sc,180))
    elif sm=="outline":
        oc=contrasting_color(dc,80)
        for dx in [-2,-1,0,1,2]:
            for dy in [-2,-1,0,1,2]:
                if dx or dy: draw.text((cx+dx,cy+dy),digit_str,font=fnt,fill=(*oc,200))
    draw.text((cx,cy),digit_str,font=fnt,fill=(*dc,255))

    angle=random.uniform(-45,45)
    if random.random()<0.1: angle=random.uniform(160,200)
    layer=layer.rotate(angle,resample=Image.BICUBIC)

    if random.random()<0.4:
        w,h=layer.size
        layer=layer.transform(layer.size,Image.PERSPECTIVE,
                              _persp_coeffs(w,h,random.uniform(0.05,0.15)),Image.BICUBIC)

    sc=random.uniform(0.7,1.3)
    layer=layer.resize((int(layer.width*sc),int(layer.height*sc)),Image.LANCZOS)
    lw,lh=layer.size
    layer=layer.crop(((lw-sz)//2,(lh-sz)//2,(lw-sz)//2+sz,(lh-sz)//2+sz))

    bg=bg.convert("RGBA"); bg.paste(layer,(0,0),layer); img=bg.convert("RGB")

    nm=random.choices(["none","gaussian","speckle","salt_pepper"],weights=[40,30,15,15])[0]
    if nm=="gaussian":
        a=np.array(img,dtype=np.float32)
        img=Image.fromarray(np.clip(a+np.random.normal(0,random.uniform(3,20),a.shape),0,255).astype(np.uint8))
    elif nm=="speckle":
        a=np.array(img,dtype=np.float32)
        img=Image.fromarray(np.clip(a*np.random.normal(1,random.uniform(0.05,0.2),a.shape),0,255).astype(np.uint8))
    elif nm=="salt_pepper":
        a=np.array(img); p=random.uniform(0.005,0.03); rn=np.random.rand(*a.shape[:2])
        a[rn<p/2]=0; a[rn>1-p/2]=255; img=Image.fromarray(a)

    if random.random()<0.5: img=ImageEnhance.Brightness(img).enhance(random.uniform(0.6,1.5))
    if random.random()<0.4: img=ImageEnhance.Contrast(img).enhance(random.uniform(0.7,1.8))
    if random.random()<0.3: img=ImageEnhance.Sharpness(img).enhance(random.uniform(0.3,3.0))
    if random.random()<0.2: img=img.filter(ImageFilter.GaussianBlur(random.uniform(0.3,1.5)))
    if random.random()<0.25:
        buf=io.BytesIO(); img.save(buf,"JPEG",quality=random.randint(40,85))
        buf.seek(0); img=Image.open(buf).copy()
    return img

# ─── Потокобезопасный счётчик ─────────────────────────────────────────────────

_lock = threading.Lock()
_done = 0

def _generate_batch(digit, start, end, out_dir, sz):
    global _done
    folder = os.path.join(out_dir, str(digit))
    count = 0
    for i in range(start, end):
        try:
            render_digit(str(digit), sz).save(
                os.path.join(folder, f"{digit}_{i:05d}.png"), "PNG")
            count += 1
        except Exception as e:
            log(f"ОШИБКА: digit={digit} i={i}: {e}")
    with _lock:
        _done += count
    return count

# ─── main ─────────────────────────────────────────────────────────────────────

def main():
    global FONTS, _done

    log("Старт")

    FONTS = _find_fonts()
    log(f"Шрифтов: {len(FONTS)}")

    total = IMAGES_PER_CLASS * 10
    log(f"Потоков: {NUM_WORKERS} | Изображений: {total:,}")

    for d in range(10):
        os.makedirs(os.path.join(OUTPUT_DIR, str(d)), exist_ok=True)
    log("Папки созданы")

    BATCH = 50
    tasks = []
    for d in range(10):
        for s in range(0, IMAGES_PER_CLASS, BATCH):
            tasks.append((d, s, min(s + BATCH, IMAGES_PER_CLASS), OUTPUT_DIR, IMG_SIZE))
    log(f"Задач: {len(tasks)} (по {BATCH} картинок)")

    # Фоновый поток для прогресса — печатает каждые 5 секунд
    t0 = time.time()
    _done = 0
    stop_ev = threading.Event()

    def progress():
        while not stop_ev.is_set():
            with _lock:
                d = _done
            elapsed = time.time() - t0
            spd = d / elapsed if elapsed > 0 else 0
            eta = (total - d) / spd if spd > 0 else 0
            log(f"Прогресс: {d:>6}/{total}  {d/total*100:5.1f}%  {spd:5.0f} img/s  ETA {eta:.0f}s")
            stop_ev.wait(5)

    pt = threading.Thread(target=progress, daemon=True)
    pt.start()

    log("Запуск генерации...")
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = [pool.submit(_generate_batch, *t) for t in tasks]
        for f in as_completed(futures):
            try:
                f.result()
            except Exception as e:
                log(f"ОШИБКА В ПОТОКЕ: {e}")

    stop_ev.set()
    pt.join()

    elapsed = time.time() - t0
    log(f"Готово! {_done:,} картинок за {elapsed:.1f}с ({_done/elapsed:.0f} img/s)")
    log(f"Датасет: {os.path.abspath(OUTPUT_DIR)}")

if __name__ == "__main__":
    main()

[07:37:03] Старт
[07:37:03] Шрифтов: 79
[07:37:03] Потоков: 10 | Изображений: 50,000
[07:37:03] Папки созданы
[07:37:03] Задач: 1000 (по 50 картинок)
[07:37:03] Прогресс:      0/50000    0.0%      0 img/s  ETA 0s
[07:37:03] Запуск генерации...
[07:37:08] Прогресс:    500/50000    1.0%    100 img/s  ETA 496s
[07:37:13] Прогресс:   1050/50000    2.1%    105 img/s  ETA 467s
[07:37:18] Прогресс:   2000/50000    4.0%    133 img/s  ETA 360s
[07:37:23] Прогресс:   2600/50000    5.2%    130 img/s  ETA 365s
[07:37:28] Прогресс:   3500/50000    7.0%    140 img/s  ETA 332s
[07:37:33] Прогресс:   4100/50000    8.2%    137 img/s  ETA 336s
[07:37:38] Прогресс:   4900/50000    9.8%    140 img/s  ETA 323s
[07:37:43] Прогресс:   5450/50000   10.9%    136 img/s  ETA 327s
[07:37:48] Прогресс:   6200/50000   12.4%    138 img/s  ETA 318s
[07:37:53] Прогресс:   6900/50000   13.8%    138 img/s  ETA 313s
[07:37:58] Прогресс:   7550/50000   15.1%    137 img/s  ETA 310s
[07:38:03] Прогресс:   8400/50000   16.8%